# xRouteBench: all-candidate official-matrix baseline

This notebook establishes the first trustworthy experiment before modifying the benchmark. It preserves the official `llmrouter_generic` train/test split, uses all 18 candidates, creates duplicate-safe validation from official training data, and evaluates fixed, random, category, learned, and oracle policies under the same quality-cost objective.

**Router inputs:** query text and inference-safe surface features only. Ground truth, responses, task labels, model outcomes, costs, and latency are labels/evaluation data and never enter the learned router input.

## 1. Reproducible configuration

The primary objective follows the benchmark's performance-cost framing. Cost is explicitly normalized by the training-only 95th percentile; latency is reported but is not optimized in this first baseline.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display

candidates_for_root = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in candidates_for_root if (path / 'data/raw').exists())
EXPERIMENT_DIR = REPO_ROOT / 'experiments/xroutebench_official_baseline'
sys.path.insert(0, str(EXPERIMENT_DIR))

from baseline import (
    TRADEOFFS, assert_no_prompt_overlap, audit_long_table, build_query_matrix,
    cheapest_fixed_choice, deterministic_grouped_validation_split, evaluate_choices,
    file_sha256, fit_router, fit_task_lookup, fixed_model_choice, hashed_text_features,
    load_local_data, predict_outcomes, prepare_long_table, task_breakdown, utility_matrix,
)

SEED = 42
VALIDATION_FRACTION = 0.20
HASH_DIMENSION = 4096
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print({'repo_root': str(REPO_ROOT), 'device': DEVICE, 'torch': torch.__version__})

## 2. Load and fingerprint the real dataset

Fingerprints make it clear which exact local Parquet artifacts produced a result.

In [ ]:
raw_train, raw_test, candidates, fingerprints = load_local_data(REPO_ROOT)
display(pd.DataFrame(fingerprints).T)

train_long = prepare_long_table(raw_train, candidates, 'train')
test_long = prepare_long_table(raw_test, candidates, 'test')
audit = pd.DataFrame([
    audit_long_table(train_long, 'official_train'),
    audit_long_table(test_long, 'official_test'),
])
display(audit)

### Why `embedding_id`, not `task_id`

`task_id` is absent or repeated for several benchmark tasks. Within each official split, `embedding_id` identifies the query instance and every ID has exactly one row for every candidate. Query hashes are used separately to keep exact duplicate prompts together when deriving validation.

In [ ]:
models = tuple(candidates['model_name'].tolist())
train_matrix = build_query_matrix(train_long, models)
test_matrix = build_query_matrix(test_long, models)

train_idx, validation_idx = deterministic_grouped_validation_split(
    train_matrix.meta, validation_fraction=VALIDATION_FRACTION, seed=SEED
)
assert_no_prompt_overlap(train_matrix.meta.iloc[train_idx], train_matrix.meta.iloc[validation_idx])
assert_no_prompt_overlap(train_matrix.meta, test_matrix.meta)

split_summary = pd.concat([
    train_matrix.meta.iloc[train_idx].assign(derived_split='train'),
    train_matrix.meta.iloc[validation_idx].assign(derived_split='validation'),
    test_matrix.meta.assign(derived_split='official_test'),
]).groupby(['derived_split', 'task_name']).size().unstack(fill_value=0)
display(split_summary)
print({'train_queries': len(train_idx), 'validation_queries': len(validation_idx), 'test_queries': len(test_matrix.meta)})

## 3. Build inference-safe query features

The first learned router deliberately avoids an external embedding download. It uses deterministic hashed unigrams/bigrams plus basic surface features. This gives us a fast, reproducible System-1 baseline that can later be replaced by a frozen encoder without changing the evaluation protocol.

In [ ]:
X_train_all = hashed_text_features(train_matrix.meta['query'], HASH_DIMENSION)
X_test = hashed_text_features(test_matrix.meta['query'], HASH_DIMENSION)

cost_scale = float(np.percentile(train_matrix.cost_usd[train_idx], 95))
latency_scale = float(np.percentile(train_matrix.latency_s[train_idx], 95))
print({
    'train_feature_shape': X_train_all.shape,
    'test_feature_shape': X_test.shape,
    'cost_p95_usd': cost_scale,
    'latency_p95_s': latency_scale,
})

## 4. Train one multi-outcome System-1 router

The router predicts all 18 models' quality, normalized cost, and normalized latency. Validation is used for early stopping. The quality-cost tradeoff is applied only at decision time, allowing the same trained router to be evaluated across the benchmark weight sweep.

In [ ]:
router, history = fit_router(
    X_train_all, train_matrix, train_idx, validation_idx,
    cost_scale=cost_scale, latency_scale=latency_scale,
    seed=SEED, device=DEVICE,
)
display(history.tail(10))
print({'epochs_run': len(history), 'best_validation_loss': history['validation_loss'].min()})

In [ ]:
pred_quality_test, pred_cost_test, pred_latency_test = predict_outcomes(
    router, X_test, device=DEVICE
)
print(pred_quality_test.shape, pred_cost_test.shape, pred_latency_test.shape)

## 5. Evaluate every policy under the same objective

For each `(alpha, beta)`, model selection baselines are fitted using train or validation only. The oracle is computed on test solely as an upper bound. The known-task lookup is intentionally strong: it receives the benchmark task name and therefore measures how much category-only routing can accomplish.

In [ ]:
rng = np.random.default_rng(SEED)
random_choices = rng.integers(0, len(models), size=len(test_matrix.meta))
cheapest_idx = cheapest_fixed_choice(train_matrix, train_idx)
results = []
learned_choices_by_tradeoff = {}

for alpha, beta in TRADEOFFS:
    best_fixed_idx = fixed_model_choice(
        train_matrix, validation_idx, alpha, beta, cost_scale
    )
    task_mapping = fit_task_lookup(
        train_matrix, train_idx, alpha, beta, cost_scale
    )
    task_choices = np.array([task_mapping[t] for t in test_matrix.meta['task_name']], dtype=np.int64)
    predicted_utility = alpha * pred_quality_test - beta * pred_cost_test
    learned_choices = predicted_utility.argmax(axis=1)
    learned_choices_by_tradeoff[(alpha, beta)] = learned_choices
    oracle_choices = utility_matrix(test_matrix, alpha, beta, cost_scale).argmax(axis=1)

    policies = [
        ('Cheapest fixed (train)', np.full(len(test_matrix.meta), cheapest_idx)),
        ('Best fixed (validation)', np.full(len(test_matrix.meta), best_fixed_idx)),
        ('Uniform random (seeded)', random_choices),
        ('Known-task lookup (train)', task_choices),
        ('Learned System-1 router', learned_choices),
        ('Per-query oracle', oracle_choices),
    ]
    for name, choices in policies:
        results.append(evaluate_choices(
            name, test_matrix, choices, alpha, beta, cost_scale
        ))

results_df = pd.DataFrame(results)
display(results_df.round(5))

## 6. Inspect the balanced tradeoff by task

Micro quality is influenced by benchmark task sizes, so the main table includes macro task quality. This cell exposes per-task behavior for the middle `(0.6, 0.4)` setting.

In [ ]:
FOCUS_TRADEOFF = (0.6, 0.4)
focus_choices = learned_choices_by_tradeoff[FOCUS_TRADEOFF]
display(task_breakdown(
    test_matrix, focus_choices, *FOCUS_TRADEOFF, cost_scale
).round(5))

route_counts = (
    pd.Series([models[index] for index in focus_choices])
    .value_counts().rename('routes').rename_axis('model').to_frame()
)
route_counts['share'] = route_counts['routes'] / route_counts['routes'].sum()
display(route_counts)

## 7. Baseline contract and next modifications

This run is the reference point. A later modification should keep the same official test matrix, input-leakage rules, objective definition, normalization scale, and baselines unless the experiment explicitly studies one of those choices.

Good next experiments, one change at a time:

1. replace hashing with a frozen sentence encoder;
2. compare pointwise outcome prediction with pairwise ranking;
3. add a latency penalty and a matching latency-aware oracle;
4. select smaller candidate pools using training-only Pareto coverage;
5. calibrate on real application traffic or new live APIs.